# Phase 2: Data Cleaning & Feature Engineering
**Project:** Mirae Asset Analytics  
**Objective:** Transform raw data into a clean, analysis-ready user-level dataset with engineered features for churn prediction, segmentation, and revenue analysis.

---
## Step 1: Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load all raw datasets
def find_project_root(start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return str(candidate)
    return str(current.parent if current.name == 'notebooks' else current)

BASE = find_project_root()
users        = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'users.csv'))
sessions     = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'sessions.csv'))
transactions = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'transactions.csv'))
events       = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'events.csv'))
campaigns    = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'campaigns.csv'))
# Convert all date columns to datetime
users['signup_date']               = pd.to_datetime(users['signup_date'])
sessions['session_date']           = pd.to_datetime(sessions['session_date'])
transactions['transaction_date']   = pd.to_datetime(transactions['transaction_date'])
events['event_date']               = pd.to_datetime(events['event_date'])

if 'is_registered' not in users.columns:
    users['is_registered'] = 1
registered_users = users[users['is_registered'].fillna(1).astype(int).eq(1)].copy()

print('All datasets loaded successfully.')
print(f'Visitor/user rows: {users.shape} | Registered users: {registered_users.shape}')
print(f'Sessions: {sessions.shape} | Transactions: {transactions.shape}')
print(f'Events: {events.shape} | Campaigns: {campaigns.shape}')

All datasets loaded successfully.
Visitor/user rows: (12500, 9) | Registered users: (10000, 9)
Sessions: (50000, 5) | Transactions: (15000, 5)
Events: (90000, 4) | Campaigns: (50, 4)


---
## Step 2: Data Quality Checks
Before building any features, we verify table shape, nulls, duplicates, and key integrity across all five raw inputs. This is the first production checkpoint: only clean and internally consistent data should move into the user-level analytics layer.


In [2]:
for name, df in [('users', users), ('sessions', sessions),
                 ('transactions', transactions), ('events', events),
                 ('campaigns', campaigns)]:
    nulls = df.isnull().sum().sum()
    dups  = df.duplicated().sum()
    print(f'{name:15s} | shape: {str(df.shape):18s} | nulls: {nulls} | duplicates: {dups}')

users           | shape: (12500, 9)         | nulls: 0 | duplicates: 0
sessions        | shape: (50000, 5)         | nulls: 0 | duplicates: 0
transactions    | shape: (15000, 5)         | nulls: 0 | duplicates: 0
events          | shape: (90000, 4)         | nulls: 0 | duplicates: 0
campaigns       | shape: (50, 4)            | nulls: 0 | duplicates: 0


**Observation:** Since the synthetic data is generated with `np.random.seed(42)`, the expected baseline is zero nulls and zero duplicates across all raw tables. Any deviation would be treated as a data-quality exception before feature engineering, modelling, or dashboard reporting.


---
## Step 3: Validate Temporal Logic
A session, transaction, or funnel event that occurs *before* a user's signup date is a data integrity violation. Temporal consistency is a core data contract: behaviour must occur on or after signup before it is used for features, cohorts, funnel metrics, or model labels.


In [3]:
# Merge signup date into behavioural tables
sessions_check = sessions.merge(users[['user_id', 'signup_date']], on='user_id', how='left')
transactions_check = transactions.merge(users[['user_id', 'signup_date']], on='user_id', how='left')
events_check = events.merge(users[['user_id', 'signup_date']], on='user_id', how='left')

# Flag records where activity precedes signup
bad_sessions = sessions_check[sessions_check['session_date'] < sessions_check['signup_date']]
bad_transactions = transactions_check[transactions_check['transaction_date'] < transactions_check['signup_date']]
bad_events = events_check[events_check['event_date'] < events_check['signup_date']]

print(f'Sessions before signup date     : {len(bad_sessions):,}')
print(f'Transactions before signup date : {len(bad_transactions):,}')
print(f'Events before signup date       : {len(bad_events):,}')

# Production safeguard: enforce the temporal contract before aggregation.
# In a valid run, these counts are zero and no rows are removed.
sessions = sessions_check[sessions_check['session_date'] >= sessions_check['signup_date']].drop(columns='signup_date')
transactions = transactions_check[transactions_check['transaction_date'] >= transactions_check['signup_date']].drop(columns='signup_date')
events = events_check[events_check['event_date'] >= events_check['signup_date']].drop(columns='signup_date')

print()
print(f'Cleaned sessions     : {len(sessions):,}')
print(f'Cleaned transactions : {len(transactions):,}')
print(f'Cleaned events       : {len(events):,}')


Sessions before signup date     : 0
Transactions before signup date : 0
Events before signup date       : 0

Cleaned sessions     : 50,000
Cleaned transactions : 15,000
Cleaned events       : 90,000


---
## Step 4: Build User-Level Aggregations
The goal is one row per user — a single flat table that consolidates behavioural signals from sessions, transactions, and events. This is the format all downstream analyses (churn model, segmentation, Power BI) will consume.

In [4]:
# --- Session metrics ---
session_agg = sessions.groupby('user_id').agg(
    total_sessions      = ('session_id', 'count'),
    avg_session_duration= ('duration_minutes', 'mean'),
    total_pages_viewed  = ('pages_viewed', 'sum'),
    last_active_date    = ('session_date', 'max')
).reset_index()

# --- Transaction metrics ---
txn_agg = transactions.groupby('user_id').agg(
    total_revenue        = ('amount', 'sum'),
    total_purchases      = ('transaction_id', 'count'),
    avg_order_value      = ('amount', 'mean'),
    first_purchase_date  = ('transaction_date', 'min'),
    last_purchase_date   = ('transaction_date', 'max')
).reset_index()

print('Session aggregation  :', session_agg.shape)
print('Transaction aggregation:', txn_agg.shape)

Session aggregation  : (9934, 5)
Transaction aggregation: (4800, 6)


---
## Step 5: Merge Into Master User Dataset

In [5]:
# Left join so every registered user is retained even if they have no sessions or purchases
user_data = registered_users.copy()
user_data = user_data.merge(session_agg, on='user_id', how='left')
user_data = user_data.merge(txn_agg,     on='user_id', how='left')

# Fill numeric columns with 0 for users with no activity
numeric_cols = ['total_sessions', 'avg_session_duration', 'total_pages_viewed',
                'total_revenue', 'total_purchases', 'avg_order_value']
user_data[numeric_cols] = user_data[numeric_cols].fillna(0)

# Validate: row count must equal registered user count
assert len(user_data) == len(registered_users), 'Row count mismatch after merge — check for duplicate keys.'
print(f'Master dataset shape : {user_data.shape}')
print(f'Users with 0 sessions    : {(user_data["total_sessions"] == 0).sum():,}')
print(f'Users with 0 purchases   : {(user_data["total_purchases"] == 0).sum():,}')
user_data.head()

Master dataset shape : (10000, 18)


Users with 0 sessions    : 66
Users with 0 purchases   : 5,200


,user_id,signup_date,country,state,device,age,gender,acquisition_channel,is_registered,total_sessions,avg_session_duration,total_pages_viewed,last_active_date,total_revenue,total_purchases,avg_order_value,first_purchase_date,last_purchase_date
0,1,2023-04-13,India,Uttar Pradesh,Mobile,45,Male,Facebook Ads,1,5.0,13.400000,31.0,2023-06-28,0.0,0.0,0.000000,NaT,NaT
1,2,2023-06-29,India,Karnataka,Mobile,47,Male,Referral,1,6.0,37.000000,60.0,2023-06-29,0.0,0.0,0.000000,NaT,NaT
2,3,2023-04-03,India,Gujarat,Mobile,58,Male,Organic,1,3.0,29.333333,28.0,2023-06-21,4154.0,3.0,1384.666667,2023-05-05,2023-06-18
3,4,2023-01-15,India,Maharashtra,Mobile,46,Female,Facebook Ads,1,6.0,25.166667,52.0,2023-06-23,0.0,0.0,0.000000,NaT,NaT
4,5,2023-04-17,India,Maharashtra,Mobile,51,Female,Google Ads,1,4.0,48.500000,18.0,2023-06-25,5142.0,2.0,2571.000000,2023-04-19,2023-05-04


---
## Step 6: Feature Engineering
Raw aggregations alone are not enough for modelling or business insight. We create derived features that capture user behaviour in ways that are directly interpretable by a business stakeholder.

In [6]:
# Reference point: the latest date in the dataset acts as 'today'
latest_date = sessions['session_date'].max()

# --- Tenure ---
# How long has this user been with the platform? Longer tenure users should churn less.
user_data['days_since_signup'] = (latest_date - user_data['signup_date']).dt.days

# --- Recency ---
# Days since last session. High recency = user drifting away.
user_data['days_since_last_active'] = (
    latest_date - user_data['last_active_date']
).dt.days.fillna(user_data['days_since_signup'])  # if never active, treat as inactive since signup

# --- Purchase efficiency ---
# Revenue per purchase — differentiates high-value occasional buyers from frequent low-value buyers.
user_data['avg_revenue_per_purchase'] = np.where(
    user_data['total_purchases'] > 0,
    user_data['total_revenue'] / user_data['total_purchases'],
    0
)

# --- Session efficiency ---
# Revenue generated per session. A proxy for conversion quality.
user_data['revenue_per_session'] = np.where(
    user_data['total_sessions'] > 0,
    user_data['total_revenue'] / user_data['total_sessions'],
    0
)

# --- Has purchased flag ---
# Binary: did the user ever convert? Useful for funnel analysis.
user_data['has_purchased'] = (user_data['total_purchases'] > 0).astype(int)

print('New features added: days_since_signup, days_since_last_active,')
print('  avg_revenue_per_purchase, revenue_per_session, has_purchased')

New features added: days_since_signup, days_since_last_active,
  avg_revenue_per_purchase, revenue_per_session, has_purchased


---
## Step 7: Engagement Score
A composite metric that summarises how active a user is across three dimensions: frequency (sessions), depth (session duration), and conversion (purchases). Weights are based on business reasoning — frequency matters most, purchases least (they are sparse by nature).

In [7]:
from sklearn.preprocessing import MinMaxScaler
from joblib import dump

MODELS_DIR = os.path.join(BASE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Calculate raw score
user_data['engagement_score_raw'] = (
    user_data['total_sessions']       * 0.5 +
    user_data['avg_session_duration'] * 0.3 +
    user_data['total_purchases']      * 0.2
)

# Normalise to 0-1 and persist the scaler used for this analytical baseline.
scaler = MinMaxScaler()
user_data['engagement_score'] = scaler.fit_transform(
    user_data[['engagement_score_raw']]
)
dump(scaler, os.path.join(MODELS_DIR, 'engagement_score_scaler.pkl'))

print(f'Engagement score - min: {user_data["engagement_score"].min():.3f}, '
      f'max: {user_data["engagement_score"].max():.3f}, '
      f'mean: {user_data["engagement_score"].mean():.3f}')
print('Saved scaler: models/engagement_score_scaler.pkl')


Engagement score - min: 0.000, max: 1.000, mean: 0.586
Saved scaler: models/engagement_score_scaler.pkl


---
## Step 8: Churn Flag
Churn is defined as 30 days of inactivity from the last known session date, but only for users whose tenure is long enough for that outcome to be observable. Users with fewer than 30 days of history are marked as not yet churn-observable, which prevents recent signups from biasing retention and cohort interpretation.

**Note:** Never-active users are flagged as churned only when they have crossed the 30-day observation window. Before that point they are tracked as activation risk, not confirmed churn.


In [8]:
CHURN_THRESHOLD_DAYS = 30  # document assumption explicitly

user_data['churn_eligible'] = (user_data['days_since_signup'] >= CHURN_THRESHOLD_DAYS).astype(int)
inactive_over_threshold = (
    user_data['last_active_date'].isna() |
    ((latest_date - user_data['last_active_date']).dt.days > CHURN_THRESHOLD_DAYS)
)
user_data['churn'] = np.where(
    user_data['churn_eligible'].eq(1) & inactive_over_threshold,
    1, 0
)

churn_rate  = user_data['churn'].mean()
never_active = user_data['last_active_date'].isna().sum()
observable = user_data['churn_eligible'].sum()

print(f'Churn threshold       : {CHURN_THRESHOLD_DAYS} days')
print(f'Churn-observable users: {observable:,}')
print(f'Overall churn rate    : {churn_rate:.1%}')
print(f'Never-active users    : {never_active:,} (churned only after observation window)')
print(f'Active users          : {(user_data["churn"]==0).sum():,}')
print(f'Churned users         : {(user_data["churn"]==1).sum():,}')


Churn threshold       : 30 days
Churn-observable users: 8,309
Overall churn rate    : 17.4%
Never-active users    : 66 (churned only after observation window)
Active users          : 8,256
Churned users         : 1,744


**Observation:** The churn baseline is commercially usable because it separates confirmed inactivity from users who are too new to judge. The 30-day inactivity rule produces the target variable for Phase 6 (Advanced Churn Analysis) and Phase 14 (Predictive Modelling), while `churn_eligible` preserves the observation-window context.


---
## Step 9: Dataset Validation
Final checks before saving — confirming shape, null counts, and key column distributions are as expected.

In [9]:
print('='*55)
print('  FINAL DATASET VALIDATION')
print('='*55)
print(f'  Shape                  : {user_data.shape}')
print(f'  Total nulls            : {user_data.isnull().sum().sum()}')
print(f'  Churn rate             : {user_data["churn"].mean():.1%}')
print(f'  Users with purchases   : {user_data["has_purchased"].sum():,} ({user_data["has_purchased"].mean():.1%})')
print(f'  Avg revenue / user     : Rs {user_data["total_revenue"].mean():,.0f}')
print(f'  Avg sessions / user    : {user_data["total_sessions"].mean():.1f}')
print(f'  Avg engagement score   : {user_data["engagement_score"].mean():.3f}')
print(f'  Date range (signups)   : {user_data["signup_date"].min().date()} to {user_data["signup_date"].max().date()}')
print('='*55)

# Show dtypes for all columns
print('\nColumn types:')
print(user_data.dtypes.to_string())

  FINAL DATASET VALIDATION
  Shape                  : (10000, 27)
  Total nulls            : 10466
  Churn rate             : 17.4%
  Users with purchases   : 4,800 (48.0%)
  Avg revenue / user     : Rs 3,810
  Avg sessions / user    : 5.0
  Avg engagement score   : 0.586
  Date range (signups)   : 2023-01-01 to 2023-06-29

Column types:
user_id                              int64
signup_date                 datetime64[ns]
country                             object
state                               object
device                              object
age                                  int64
gender                              object
acquisition_channel                 object
is_registered                        int64
total_sessions                     float64
avg_session_duration               float64
total_pages_viewed                 float64
last_active_date            datetime64[ns]
total_revenue                      float64
total_purchases                    float64
avg_order_value

---
## Step 10: Save Processed Data

In [10]:
os.makedirs(os.path.join(BASE, 'data', 'processed'), exist_ok=True)
user_data.to_csv(os.path.join(BASE, 'data', 'processed', 'user_data.csv'), index=False)

print('Cleaned dataset saved to data/processed/user_data.csv')
print(f'Final shape: {user_data.shape[0]:,} users x {user_data.shape[1]} features')
print('\nFeatures in final dataset:')
for col in user_data.columns:
    print(f'  {col}')

Cleaned dataset saved to data/processed/user_data.csv
Final shape: 10,000 users x 27 features

Features in final dataset:
  user_id
  signup_date
  country
  state
  device
  age
  gender
  acquisition_channel
  is_registered
  total_sessions
  avg_session_duration
  total_pages_viewed
  last_active_date
  total_revenue
  total_purchases
  avg_order_value
  first_purchase_date
  last_purchase_date
  days_since_signup
  days_since_last_active
  avg_revenue_per_purchase
  revenue_per_session
  has_purchased
  engagement_score_raw
  engagement_score
  churn_eligible
  churn


---
## Summary

**What this notebook produced:**

- Validated data integrity across all 5 raw tables: nulls, duplicates, key integrity, and temporal consistency
- Enforced the temporal contract that sessions, transactions, and events must occur on or after signup
- Built a registered-user master dataset by aggregating sessions and transactions
- Engineered features: `days_since_signup`, `days_since_last_active`, `avg_revenue_per_purchase`, `revenue_per_session`, `has_purchased`
- Created a normalised `engagement_score` (0-1) weighted across frequency, depth, and conversion, with the scaler persisted under `models/`
- Defined and applied a tenure-aware `churn` flag (30-day inactivity rule) with `churn_eligible` for observation-window control
- Validated the final 10,000 registered-user dataset; remaining nulls are expected date nulls for users with no session or no purchase

**Baseline produced for downstream analysis:**

- Total users: 10,000
- Total revenue: Rs 38.10M
- Conversion rate: 48.0%
- Churn rate: recalculated in the validation cell and stored in `project_metrics.json`

**Ready for:** Phase 4 (EDA), Phase 5 (Funnel Analysis), Phase 6 (Churn Deep Dive), Phase 7 (Cohort Analysis)
